# Black Swan — ทดลอง เลือกวิธี ตรวจคำตอบ และเรียนรู้
ฉบับจัดระเบียบจาก `สำเนาของ_untitled7.py` • 14 กันยายน 2026 • package 0.2.1

สมุดนี้รันชุดข้อมูลเดิม กรณีที่คุณสร้าง และโจทย์ตัวเลขใหม่ที่เข้าครบทั้ง 8 ส่วนของ Black Swan
ตัวเลขทุกตารางอ่านจากผลรันของครั้งนี้ การทดลองด้านคณิตศาสตร์ พยานหลักฐาน และโปรโตคอลมีป้ายกำกับแยกชัด

**วิธีเริ่มบนมือถือ:** อัปโหลดสมุดนี้เข้า Colab → เลือก Run all → เมื่อถูกถาม ให้อัปโหลด `Black_Swan_Colab_Review_2026-09-14.zip` ด้วย
แต่ละครั้งสร้างโฟลเดอร์ผลใหม่ จึงรันซ้ำได้โดยไม่ทับหลักฐานเดิม

เป้าหมายคือห้องทดลองการแก้ปัญหาจากหลายรูปแบบข้อมูล ชุดกราฟและ Medicare เป็นตัวอย่างโจทย์

ฉบับนี้ปรับการควบคุมการรัน: ต้องผ่านทุกเกณฑ์ก่อนใช้ผลหรือ memory และตรวจ checksum ของ ZIP ก่อนแตกไฟล์
ยังใช้ Core 0.2.1 เดิม การแก้ข้อความผิด 10 เคสและการแก้คำตอบจนแย่ลง 5 เคสยังเป็นงานถัดไป
ล้าง output เดิมแล้ว ตัวเลขจะปรากฏเมื่อคุณรันสมุดครั้งใหม่สำเร็จ


## 1. เตรียมชุดโค้ด
ใช้ Python มาตรฐานสำหรับตัวระบบ และ Matplotlib สำหรับกราฟ ไม่ต้องมี API key
บน Colab จะเปิดหน้าต่างเลือกไฟล์เมื่อยังไม่พบ ZIP ที่ต้องใช้


In [ ]:
from pathlib import Path
import hashlib, json, uuid

# This state is reset before every attempted run.
_VERIFIED_RUN = None
REQUIRED_EXPERIMENT_CHECKS = (
    "reviewed_probe_contracts", "graph_traversals_actually_executed",
    "actual_object_cycle_rejected", "next_request_recovers",
    "learning_gate_passed", "numeric_answers_correct",
    "numeric_first_choice_improves", "old_regression_answer_count_preserved",
    "old_regression_automatic_errors_not_increased",
    "memory_frozen_for_holdout", "regression_tests_passed",
)
EXPECTED_ARCHIVE_SHA256 = "730472506e90712731589d6633203bd964bdace67d67f26ba5676625b1bc8cb1"

def invalidate_verified_run():
    global _VERIFIED_RUN
    _VERIFIED_RUN = None

def validate_experiment_summary(report):
    if not isinstance(report, dict) or report.get("all_checks_passed") is not True:
        raise RuntimeError("Experiment report did not confirm success.")
    checks = report.get("checks")
    if not isinstance(checks, dict):
        raise RuntimeError("Experiment report has no check results.")
    missing = sorted(set(REQUIRED_EXPERIMENT_CHECKS) - set(checks))
    failed = sorted(name for name, value in checks.items() if value is not True)
    if missing or failed:
        raise RuntimeError("Unverified experiment checks; missing=" + str(missing)
                           + "; failed=" + str(failed))
    return report

def accept_verified_run(run_dir):
    global _VERIFIED_RUN
    invalidate_verified_run()
    run_dir = Path(run_dir).resolve()
    try:
        raw = (run_dir / "summary.json").read_bytes()
        report = validate_experiment_summary(json.loads(raw))
    except (OSError, ValueError, TypeError) as exc:
        raise RuntimeError("Experiment summary is missing or unreadable.") from exc
    _VERIFIED_RUN = (run_dir, hashlib.sha256(raw).hexdigest())
    return report

def require_verified_run():
    if _VERIFIED_RUN is None:
        raise RuntimeError("Run the experiment cell successfully before using its results or memory.")
    run_dir, summary_digest = _VERIFIED_RUN
    try:
        raw = (run_dir / "summary.json").read_bytes()
    except OSError as exc:
        raise RuntimeError("The verified experiment summary is no longer available.") from exc
    if hashlib.sha256(raw).hexdigest() != summary_digest:
        raise RuntimeError("The experiment summary changed after validation. Run again.")
    return run_dir

def write_notebook_logs(run_dir, stdout, stderr):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    for name, value in (("notebook_stdout.log", stdout),
                        ("notebook_stderr.log", stderr)):
        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")
        (run_dir / name).write_text(value or "", encoding="utf-8")

def verify_archive(archive, expected_digest=EXPECTED_ARCHIVE_SHA256):
    digest = hashlib.sha256()
    with Path(archive).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected_digest:
        raise RuntimeError("The ZIP does not match the reviewed 0.2.1 archive. Use the matching ZIP.")
    return digest.hexdigest()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, sys, subprocess, zipfile

invalidate_verified_run()
PROJECT_NAME = 'black_swan_general_decision_lab_2026_09_14'
ARCHIVE_NAME = 'Black_Swan_Colab_Review_2026-09-14.zip'
ROOT = next((p.resolve() for p in [Path.cwd(), Path.cwd()/PROJECT_NAME, Path.cwd().parent]
             if (p/'run_user_experiments.py').is_file()), None)
if ROOT is None:
    archive = Path.cwd()/ARCHIVE_NAME
    if not archive.exists():
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError('Place '+ARCHIVE_NAME+' beside the notebook.') from exc
        files.upload()
    if not archive.exists():
        raise FileNotFoundError('Required archive: '+ARCHIVE_NAME)
    verify_archive(archive)
    destination = Path.cwd()/('black-swan-session-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    with zipfile.ZipFile(archive) as bundle:
        if sum(m.file_size for m in bundle.infolist()) > 100_000_000:
            raise ValueError('Unexpected expanded archive size')
        for member in bundle.infolist():
            if not (destination/member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Invalid archive member path')
        bundle.extractall(destination)
    ROOT = (destination/PROJECT_NAME).resolve()
if not (ROOT/'run_user_experiments.py').is_file():
    raise FileNotFoundError('The reviewed project code is missing')
sys.path.insert(0,str(ROOT))
sys.path.insert(0,str(ROOT/'src'))
print('พร้อมใช้:', ROOT.name)


## 2. รันการทดลองและตรวจโค้ด
ชุดเก่า 270 เคสใช้ตรวจว่าความสามารถเดิมเสียหรือไม่ ชุดตัวเลขใหม่แยกเป็นเรียน 12 / ตรวจผ่านประตู 8 / ทดสอบ 12 เคส
ประตูการเรียนตรวจโจทย์เดิมอีก 60 เคสด้วย รวม 68 เคส คะแนน REVIEW ของชุดข้อมูลเสียหายแสดงแยกจากโจทย์ทั่วไป

หากเซลล์นี้หยุดด้วยข้อผิดพลาด ผลเก่าจะถูกยกเลิกสำหรับเซลล์ที่ใช้ memory และรายงาน
stdout/stderr จะถูกเก็บเต็มในโฟลเดอร์รอบนั้น รวมกรณีหมดเวลา
สามารถรันเซลล์ส่งออกท้ายสมุดเพื่อเก็บไฟล์วิเคราะห์ปัญหาได้ แม้การทดลองไม่ผ่าน


In [ ]:
invalidate_verified_run()
RUN_DIR = None
summary = None
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
CANDIDATE_RUN_DIR = ROOT/'runs'/('notebook-'+stamp+'-'+uuid.uuid4().hex[:8])
command = [sys.executable,str(ROOT/'run_user_experiments.py'),
           '--output',str(CANDIDATE_RUN_DIR),'--run-tests']
try:
    result = subprocess.run(command,cwd=ROOT,text=True,capture_output=True,timeout=600)
except subprocess.TimeoutExpired as exc:
    write_notebook_logs(CANDIDATE_RUN_DIR, exc.stdout, exc.stderr)
    raise RuntimeError('Experiment timed out. Logs: '+str(CANDIDATE_RUN_DIR)) from exc
write_notebook_logs(CANDIDATE_RUN_DIR, result.stdout, result.stderr)
print(result.stdout[-5000:])
if result.returncode:
    print(result.stderr[-3000:])
    raise RuntimeError('Experiment failed. Inspect summary.json and logs in '+str(CANDIDATE_RUN_DIR))
summary = accept_verified_run(CANDIDATE_RUN_DIR)
RUN_DIR = require_verified_run()
print('ผล: ผ่านเกณฑ์ที่กำหนด')
print('บันทึกผล:', RUN_DIR)


## 3. เปรียบเทียบก่อน–หลังจากผลจริง
“ตอบถูก” นับเฉพาะโจทย์ที่มีคำตอบให้ตรวจ การส่งขอข้อมูลหรือหยุดได้ถูกกรณีไม่นับว่าแก้โจทย์สำเร็จ
“เลือกครั้งแรกถูก” ตรวจทั้งคำตอบของวิธีแรกและผลตรวจคำตอบ ความมั่นใจในระบบไม่ใช่เปอร์เซ็นต์โอกาสถูก


In [ ]:
RUN_DIR = require_verified_run()
for title,key in [('ชุดเดิม: regression replay','old_regression'),('โจทย์ตัวเลขใหม่: ชุดแยกจากการเรียน','numerical')]:
    before,after = [summary[key][name]['metrics'] for name in ('before','after')]
    print('\n'+title)
    for label,name in [('ตอบถูก','correct_answers'),('โจทย์มีคำตอบ','answerable_cases'),('REVIEW','review_count')]:
        print(f'{label}: {before[name]} -> {after[name]}')
    print(f"เลือกครั้งแรกถูก: {before['first_attempt_success_rate']:.2%} -> {after['first_attempt_success_rate']:.2%}")
    print(f"จำนวนวิธีเฉลี่ย: {before['mean_attempts']:.2f} -> {after['mean_attempts']:.2f}")
print('\nข้อจำกัด:',summary['known_limit'])


In [ ]:
RUN_DIR = require_verified_run()
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11})
fig,axes = plt.subplots(2,1,figsize=(7,5.3),layout='constrained')
for axis,key,title in zip(axes,['old_regression','numerical'],['Existing tasks','New recurrence tasks']):
    values = [summary[key][name]['metrics']['first_attempt_success_rate']*100 for name in ('before','after')]
    axis.barh(['Before','After'],values,color=['#d6dee8','#245d97'],edgecolor='#245d97',height=.55)
    axis.set_xlim(0,100)
    counts = [summary[key][name]['metrics']['answerable_cases'] for name in ('before','after')]
    axis.set_title(title+' (answerable: '+str(counts[0])+' / '+str(counts[1])+')',loc='left')
    axis.set_xlabel('First attempt correct (%)')
    axis.invert_yaxis()
    axis.spines[['top','right']].set_visible(False)
    axis.grid(axis='x',alpha=.2)
    axis.set_axisbelow(True)
    for index,value in enumerate(values):
        axis.text(min(value+2,98),index,f'{value:.1f}%',ha='right' if value>92 else 'left',va='center',
                  color='white' if value>92 else '#18212b')
FIGURE_PATH = RUN_DIR/'first_choice.png'
fig.savefig(FIGURE_PATH,dpi=150,bbox_inches='tight')
try:
    from IPython.display import Image, display
    display(Image(filename=str(FIGURE_PATH)))
except ImportError:
    print('Figure:',FIGURE_PATH)
plt.close(fig)


## 4. ดูว่า Black Swan แก้ความผิดพลาดอย่างไร
โจทย์ recurrence ใช้สูตรเดียวกับการทดลองตัวเลขของคุณ วิธี float อาจสะสมความคลาดเคลื่อน
ตัวตรวจใช้สูตรปิดคำนวณแยกจาก Solver จึงไม่ใช้แค่เงื่อนไข “ค่าควรอยู่ใกล้ 6” รับรองคำตอบ
ค่าเมื่อเดินไป 35 ขั้นยังไม่เท่ากับลิมิต 6 พอดี


In [ ]:
RUN_DIR = require_verified_run()
from black_swan_general import BlackSwanGeneralEngine
from black_swan_general.io import load_release
from experiments.cases import recurrence_case
case = recurrence_case('show-correction','probe',35,1,1)
for title,memory_dir in [('ก่อนเรียน',ROOT/'runs/verified-final'),('หลังเรียน',RUN_DIR)]:
    engine = BlackSwanGeneralEngine(memory=load_release(memory_dir))
    decision = engine.decide(case.request)
    print('\n'+title, decision.route, 'คำตอบ:',decision.answer)
    for attempt in decision.attempts:
        print(attempt.attempt_index,attempt.ranked_strategy.strategy,attempt.verification.accepted,attempt.verification.reason_code)


## 5. ตรวจกรณีที่คุณสร้างและกราฟวนลูปจริง
`opaque-*` คงรูปแบบจากต้นฉบับเพื่อดูว่าระบบรับหรือปฏิเสธที่จุดไหน
`traversal-*` ส่ง nodes, edges, source, target และ objective ที่ตรงสัญญา จึงได้ทดสอบการเดินกราฟจริง
กราฟที่อ้างโหนดด้วยชื่อกับ Python object ที่อ้างตัวเองเป็นคนละกรณี


In [ ]:
RUN_DIR = require_verified_run()
probes = json.loads((RUN_DIR/'core_probes.json').read_text())
for row in probes['rows']:
    print(f"{row['problem_id']}: {row['route']} | attempts={row['attempts']} | contract_ok={row['correct']}")
print('\nตอบโจทย์ได้:',probes['metrics']['correct_answers'],'/',probes['metrics']['answerable_cases'])
print('กรณีทั้งหมด:',probes['metrics']['cases'],'; REVIEW:',probes['metrics']['review_count'])
cycle = json.loads((RUN_DIR/'object_cycle.json').read_text())
print('Python object cycle:',cycle['decision']['reason_code'])
print('คำขอถัดไป:',cycle['next_decision']['route'])


## 6. เปลี่ยนข้อมูลทดลองเอง
แก้ USER_REQUEST แล้วรันเซลล์นี้ใหม่ได้ `payload` คือข้อมูลที่ต้องการแก้โจทย์
ตัวอย่างนี้นับจำนวนสมาชิก ไม่ใช่ทำนายเลขถัดไป ถ้าต้องการทดลองแนวโน้ม ใช้ `forecast_next` กับ `payload={"values":[2,4,6,8,10]}`
การ solve อย่างเดียวไม่เขียน memory การเรียนต้องมีผลยืนยันและผ่านประตูแยกต่างหาก


In [ ]:
RUN_DIR = require_verified_run()
from dataclasses import asdict
from black_swan_general.io import request_from_dict
USER_REQUEST = {'problem_id':'my-sequence','objective':'count_items','payload':[2,4,6,8,10],
                'constraints':{},'risk_level':'low'}
request = request_from_dict(USER_REQUEST)
decision = BlackSwanGeneralEngine(memory=load_release(RUN_DIR)).decide(request)
print(json.dumps({'route':decision.route,'answer':decision.answer,'strategy':decision.selected_strategy,
                  'attempts':len(decision.attempts),'reason':decision.reason_code},ensure_ascii=False,indent=2))


## 7. การทดลองประกอบจากไฟล์เดิม
ส่วนนี้ใช้ฟังก์ชันใน `experiments/supporting.py` ไม่ได้ใช้หรือฝึก Black Swan Core
- Buffon: กำหนด seed และช่วงประมาณความไม่แน่นอน ใช้สูตรเฉพาะเข็มยาวไม่เกินระยะเส้น
- Ramanujan: เปรียบเทียบกับค่าอ้างอิงเต็มและแสดง error ทุกพจน์
- Subset sum: ตรวจทั้งสมาชิก จำนวนครั้ง และผลรวม; จำกัดจำนวนการค้นหา
- พยานหลักฐาน: แยกเจ้าของเครื่องออกจากผู้ใช้เครื่องขณะเกิดเหตุ
- Protocol: ผู้รับเห็นเฉพาะข้อมูลรหัส ไม่มี mode หรือเฉลยส่งให้ผู้รับ; วัดความสำเร็จและตัวควบคุม ไม่สรุปว่าโมเดลวิวัฒนาการ


In [ ]:
RUN_DIR = require_verified_run()
support = json.loads((RUN_DIR/'supporting_experiments.json').read_text())
buffon_result = support['buffon']
print('Buffon:',buffon_result['estimate'],'ช่วงประมาณ 95%:',buffon_result['approx_95pct_interval'])
print('Ramanujan:',support['ramanujan']['rows'][-1])
subset = support['subset']
print('Subset ตัวอย่างที่ตรวจผ่าน:',sum(row['verified'] for row in subset['planted_cases']),'/',len(subset['planted_cases']))
print('Subset เมื่อเกินงบ:',subset['bounded_no_solution_case']['first']['status'],
      '-> วิธีสำรอง:',subset['bounded_no_solution_case']['alternative']['status'])
protocol_rows = support['protocol']['rows']
for key in ['success','shuffled_success','truncated_success','random_success']:
    print('Protocol',key,':',sum(row[key] for row in protocol_rows),'/',len(protocol_rows))
print('หลักฐานยังระบุตัวบุคคลไม่ได้:',support['evidence']['unresolved_personal_attribution'])


## 8. หลักฐานและการทดลองต่อไป
`summary.json` สรุปผล; `numeric_before/after.json` เก็บทุกความพยายาม; `training_outcomes.jsonl` เก็บผลที่ยืนยันแล้ว
โจทย์และเฉลยแยกไฟล์ใน datasets; `learning_gate.json` บันทึกเหตุผลที่ยอมให้ใช้ memory

| ส่วน | ตำแหน่ง |
|---|---|
| 1 Data Interpreter | src/black_swan_general/interpreter.py |
| 2 Strategy Selector | src/black_swan_general/selector.py |
| 3 Solver | strategies.py และ numerics.py |
| 4 Verifier | verifier.py |
| 5 Correction Loop | correction.py และ runtime.py |
| 6 Outcome Memory | memory.py |
| 7 Learning Gate | learning_gate.py |
| 8 Decision Policy | policy.py |

ขั้นถัดไปคือแก้ข้อจำกัดข้อความบนชุดพัฒนา แล้วประเมินด้วยชุดสุดท้ายชุดใหม่ก่อนสรุปว่าดีขึ้น
ผลปัจจุบันรองรับการเป็นห้องทดลอง Python ยังไม่มีหลักฐานรับรองการใช้งาน production หรือการสร้างวิธีแก้ใหม่เอง


In [ ]:
# Also export diagnostic logs when the latest experiment failed.
if 'CANDIDATE_RUN_DIR' not in globals() or not CANDIDATE_RUN_DIR.is_dir():
    raise RuntimeError('No experiment output is available to export.')
EXPORT_RUN_DIR = CANDIDATE_RUN_DIR
try:
    export_verified = require_verified_run() == EXPORT_RUN_DIR.resolve()
    export_reason = 'All required checks passed; summary unchanged.'
except RuntimeError as exc:
    export_verified = False
    export_reason = str(exc)
(EXPORT_RUN_DIR/'notebook_export_status.json').write_text(
    json.dumps({'verified':export_verified,'reason':export_reason},
               ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
EVIDENCE_ZIP = EXPORT_RUN_DIR.parent/(EXPORT_RUN_DIR.name+'-results.zip')
with zipfile.ZipFile(EVIDENCE_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(EXPORT_RUN_DIR.rglob('*')):
        if path.is_file():
            bundle.write(path,str(path.relative_to(EXPORT_RUN_DIR)))
digest = hashlib.sha256(EVIDENCE_ZIP.read_bytes()).hexdigest()
Path(str(EVIDENCE_ZIP)+'.sha256').write_text(
    digest+'  '+EVIDENCE_ZIP.name+'\n',encoding='utf-8')
print('สถานะผล:', 'ผ่านเกณฑ์' if export_verified else 'ยังไม่ผ่าน — ใช้ตรวจปัญหา')
print('ดาวน์โหลดผลครั้งนี้ได้จากแถบไฟล์:',EVIDENCE_ZIP)
print('ไฟล์คำตอบที่ยังผิดในชุดเดิม: regression_after.json')
